# PRISM pipeline comparison (evidence-backed)

This is the isolated comparison successor to `notebooks/pipeline_orientation_comparison.ipynb`. It reads the machine-readable artifacts in this package; it does not run a pipeline or mutate project data. Rows marked `NOT_RUN`, `BLOCKED`, or `NOT_COMPARABLE` are shown rather than omitted.


In [ ]:
from pathlib import Path
import csv, json, os

configured_root = os.environ.get('PRISM_ARTIFACT_ROOT')
candidates = [Path(configured_root).expanduser()] if configured_root else []
candidates.extend((Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent))
ARTIFACT_ROOT = next((candidate.resolve() for candidate in candidates if (candidate / 'PIPELINE_MATRIX.csv').exists()), None)
if ARTIFACT_ROOT is None:
    raise FileNotFoundError('Set PRISM_ARTIFACT_ROOT to the comparison artifact directory.')
DATA = {}
for name in ('PIPELINE_MATRIX.csv', 'PERFORMANCE_COMPARISON.csv', 'RANKING_COMPARISON.csv'):
    with (ARTIFACT_ROOT / name).open(newline='', encoding='utf-8') as fh:
        DATA[name] = list(csv.DictReader(fh))
PROVENANCE = {}
for name in ('manifest.json', 'TEMPLATE_PANELS.json', 'NO_DROP_SUMMARY.json', 'RECONCILIATION.json'):
    path = ARTIFACT_ROOT / name
    if path.exists():
        PROVENANCE[name] = json.loads(path.read_text(encoding='utf-8'))
print({name: len(rows) for name, rows in DATA.items()})
print('provenance artifacts:', sorted(PROVENANCE))


In [ ]:
def rows(name, **filters):
    result = DATA[name]
    for key, value in filters.items():
        if value not in (None, '', 'ALL'):
            if key == 'outcome':
                if value == 'successful':
                    result = [r for r in result if r.get('status') == 'COMPLETED' and not r.get('failure_reason')]
                elif value == 'failed_or_missing':
                    result = [r for r in result if r.get('status') != 'COMPLETED' or r.get('failure_reason')]
            else:
                result = [r for r in result if r.get(key) == str(value)]
    return result

def applicable_filters(name, filters):
    fields = set(DATA[name][0]) if DATA[name] else set()
    return {key: value for key, value in filters.items() if key == 'outcome' or key in fields}

def funnel(record):
    return {key: record.get(key, '') for key in ('candidate_input_count', 'alignment_count', 'transformed_count', 'selected_count', 'refined_count', 'scoreable_count')}

# Parameterized filters; set values to 'ALL' to remove a filter.
FILTERS = {'template_panel': 'ALL', 'query_set': 'ALL', 'case_id': 'ALL', 'alignment_provider': 'ALL', 'surface_method': 'ALL', 'ranking_method': 'ALL', 'top_k': 'ALL', 'refinement_backend': 'ALL', 'status': 'ALL', 'outcome': 'ALL'}
selected = rows('PIPELINE_MATRIX.csv', **FILTERS)
print('selected rows:', len(selected))


In [ ]:
# Stage funnel and provenance warning.
for record in selected[:20]:
    print(record['comparison_id'], record['status'], funnel(record), record['comparability'])
print('WARNING: comparisons are causal only when panel, queries, filters, evaluator, refiner, and provenance match.')


In [ ]:
# Interactive controls backed only by the stored CSV artifacts.
try:
    import ipywidgets as widgets
    from IPython.display import display
    FILTER_FIELDS = ('template_panel', 'query_set', 'case_id', 'alignment_provider',
                     'surface_method', 'ranking_method', 'top_k',
                     'refinement_backend', 'status', 'outcome')
    def widget_options(field):
        if field == 'outcome':
            return ['ALL', 'successful', 'failed_or_missing']
        values = sorted({record.get(field, '') for record in DATA['PIPELINE_MATRIX.csv'] if record.get(field, '')})
        return ['ALL'] + values
    controls = {field: widgets.Dropdown(options=widget_options(field), value='ALL', description=field.replace('_', ' ')[:18])
               for field in FILTER_FIELDS}
    def show_filtered(**values):
        selected_rows = rows('PIPELINE_MATRIX.csv', **values)
        print('selected rows:', len(selected_rows))
        for record in selected_rows[:20]:
            print(record['comparison_id'], record['status'], funnel(record), record['comparability'])
        ranking_rows = rows('RANKING_COMPARISON.csv', **applicable_filters('RANKING_COMPARISON.csv', values))
        print('ranking rows:', len(ranking_rows))
        performance_rows = rows('PERFORMANCE_COMPARISON.csv', **applicable_filters('PERFORMANCE_COMPARISON.csv', values))
        print('performance rows:', len(performance_rows))
        print('WARNING: causal comparisons require matching panel, queries, filters, evaluator, refiner, and provenance.')
    display(widgets.VBox([widgets.HBox(list(controls.values())[i:i + 2]) for i in range(0, len(controls), 2)]))
    display(widgets.interactive_output(show_filtered, controls))
except ImportError:
    print('ipywidgets unavailable; use FILTERS above for portable parameterized analysis.')


In [ ]:
# Dependency-light comparison views. Values are parsed from the stored artifacts.
NUMERIC_FIELDS = ('template_count', 'initialization_seconds', 'alignment_seconds',
                  'transformation_seconds', 'ranking_seconds', 'refinement_seconds',
                  'total_wall_seconds', 'templates_per_second', 'candidate_input_count',
                  'alignment_count', 'transformed_count', 'selected_count', 'refined_count',
                  'scoreable_count', 'success_count', 'missing_count', 'failed_count',
                  'clash_rejection_count', 'dockq_mean', 'dockq_best', 'irmsd_mean',
                  'candidates_entering_ranking', 'candidates_forwarded', 'ranking_overhead_seconds',
                  'refinement_jobs_avoided', 'refinement_compute_saved_seconds',
                  'total_wall_seconds', 'dockq_mean', 'dockq_best', 'irmsd_mean', 'ranking_regret')
def number(value):
    try:
        return float(value) if value not in (None, '', 'UNKNOWN', 'NOT_RUN', 'NOT_MEASURED') else None
    except (TypeError, ValueError):
        return None
def view(record, fields):
    return {field: record.get(field, '') for field in fields}
stage_fields = ('comparison_id', 'status', 'template_panel', 'alignment_provider',
                'candidate_input_count', 'alignment_count', 'transformed_count',
                'selected_count', 'refined_count', 'scoreable_count',
                'initialization_seconds', 'alignment_seconds', 'transformation_seconds',
                'ranking_seconds', 'refinement_seconds', 'total_wall_seconds',
                'templates_per_second', 'success_count', 'missing_count', 'failed_count',
                'clash_rejection_count', 'comparability')
print('STAGE_FUNNEL_RUNTIME_AVAILABILITY')
for record in selected:
    print(view(record, stage_fields))
print('RANKING_REDUCTION_QUALITY')
for record in rows('RANKING_COMPARISON.csv', **applicable_filters('RANKING_COMPARISON.csv', FILTERS)):
    print(view(record, ('comparison_id', 'status', 'ranking_method', 'top_k',
                     'candidates_entering_ranking', 'candidates_forwarded',
                     'ranking_overhead_seconds', 'refinement_jobs_avoided',
                     'refinement_compute_saved_seconds', 'total_wall_seconds',
                     'dockq_mean', 'dockq_best', 'irmsd_mean', 'ranking_regret', 'quality_scope', 'failure_reason')))
print('PROVENANCE_WARNING: rows are directly comparable only when query, panel hash,\nsurface, thresholds, ranking, refiner, evaluator, revision, and stage manifests match.')


In [ ]:
# Optional figures: all values come from stored CSV rows.
try:
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display
    def coerce_numeric(frame, columns):
        for column in columns:
            if column not in frame:
                frame[column] = pd.Series(index=frame.index, dtype='float64')
            frame[column] = pd.to_numeric(frame[column], errors='coerce')
        return frame
    matrix = pd.DataFrame(rows('PIPELINE_MATRIX.csv', **FILTERS))
    for col in ('template_panel', 'alignment_provider'):
        if col not in matrix:
            matrix[col] = ''
    coerce_numeric(matrix, ('template_count', 'candidate_input_count', 'alignment_count', 'transformed_count', 'selected_count', 'refined_count', 'scoreable_count', 'success_count', 'missing_count', 'failed_count', 'clash_rejection_count', 'total_wall_seconds', 'dockq_mean', 'dockq_best', 'irmsd_mean'))
    stage_cols = ['candidate_input_count', 'alignment_count', 'transformed_count', 'selected_count', 'refined_count', 'scoreable_count']
    display(matrix.groupby(['template_panel', 'alignment_provider'], dropna=False)[stage_cols].sum(min_count=1).reset_index())
    performance = pd.DataFrame(rows('PERFORMANCE_COMPARISON.csv', **applicable_filters('PERFORMANCE_COMPARISON.csv', FILTERS)))
    coerce_numeric(performance, ('template_count', 'initialization_seconds', 'alignment_seconds', 'transformation_seconds', 'ranking_seconds', 'refinement_seconds', 'total_wall_seconds', 'baseline_seconds', 'speedup', 'templates_per_second'))
    if not performance.empty:
        performance.dropna(subset=['total_wall_seconds']).plot.barh(x='comparison_id', y='total_wall_seconds', legend=False, figsize=(10, 6), title='total wall time')
        plt.xlabel('total wall time (s)'); plt.tight_layout(); plt.show()
    runtime_cols = ['initialization_seconds', 'alignment_seconds', 'transformation_seconds', 'ranking_seconds', 'refinement_seconds']
    runtime = performance.dropna(subset=['total_wall_seconds']).set_index('comparison_id')[runtime_cols]
    if not runtime.empty:
        runtime.plot.bar(stacked=True, figsize=(12, 5), title='runtime by stage')
        plt.ylabel('seconds'); plt.tight_layout(); plt.show()
    throughput = performance.dropna(subset=['templates_per_second'])
    if not throughput.empty:
        throughput.plot.scatter(x='template_count', y='templates_per_second', figsize=(8, 5), title='template-scale throughput')
        plt.tight_layout(); plt.show()
    quality = matrix.dropna(subset=['dockq_mean'])
    if not quality.empty:
        quality.plot.scatter(x='total_wall_seconds', y='dockq_mean', figsize=(8, 5), title='speed versus DockQ (stored quality rows only)')
        plt.show()
    ranking = pd.DataFrame(rows('RANKING_COMPARISON.csv', **applicable_filters('RANKING_COMPARISON.csv', FILTERS)))
    coerce_numeric(ranking, ('template_count', 'candidates_entering_ranking', 'candidates_forwarded', 'ranking_overhead_seconds', 'refinement_jobs_avoided', 'refinement_compute_saved_seconds', 'total_wall_seconds', 'baseline_wall_seconds', 'dockq_mean', 'dockq_best', 'irmsd_mean', 'ranking_regret'))
    ranking_numeric = ranking.dropna(subset=['candidates_entering_ranking', 'candidates_forwarded'])
    if not ranking_numeric.empty:
        ranking_numeric.plot.scatter(x='candidates_entering_ranking', y='candidates_forwarded', figsize=(7, 5), title='ranking candidate-load reduction')
        plt.tight_layout(); plt.show()
except Exception as exc:
    print(f'Optional pandas/matplotlib figures unavailable ({type(exc).__name__}: {exc}); dependency-light tables and filters remain available.')


## Preserved orientation protocol

The original orientation-comparison notebook is preserved by reference, not embedded in this report view. Its source path, SHA-256, and cell count are recorded below and in `manifest.json`; execution requires the project environment and a compute-capable notebook kernel.


Original source: `/scratch/rshadi25/GitHub/PRISM-prescript/notebooks/pipeline_orientation_comparison.ipynb`

Source SHA-256: `9f1f3511ad7dde0b23905d504611d144f5a84dbeedce27e03ae333888272d911`
Original cells retained by reference: 28.
